# Fine Tuning Heads

> Not the talking heads

In [ ]:
#| default_exp heads

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F

from torch import nn
import math
from sleepjepa.utils import trunc_normal_
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.nn.attention import SDPBackend, sdpa_kernel
from sleepjepa.jepa import MLP, JEPABlock
from sleepjepa.nested import unflatten_dim_from_batch, flatten_dim_to_batch

## Linear Probing and Fine Tuning Heads

In [ ]:
#| export
class RNNProbingHead(nn.Module):
    def __init__(self, 
                 c_in, 
                 input_size, 
                 hidden_size, 
                 n_classes,
                 missing_channel_indices=None,
                 module='GRU', 
                 rnn_dropout=0., 
                 num_rnn_layers=1,
                 pool='average',
                 predict_every_n_patches=1,
                 bidirectional=True, 
                 affine=False, 
                 pre_norm=True,
                 mlp_final_head=False,
                 linear_dropout=0.
                 ):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.c_in = c_in
        self.rnn_dropout = rnn_dropout
        self.bidirectional = bidirectional
        self.n_classes = n_classes
        self.pool = pool
        self.num_rnn_layers = num_rnn_layers
        self.predict_every_n_patches = predict_every_n_patches
        self.affine = affine
        self.missing_channel_indices = missing_channel_indices



        if pre_norm:
            self.norm_layer = nn.LayerNorm(self.input_size)
        else:
            self.norm_layer = nn.Identity()

        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(c_in))
            self.affine_bias = nn.Parameter(torch.zeros(c_in))

        self.flatten = nn.Flatten(start_dim=-2, end_dim=-1)

        if module.lower() == 'gru':
            self.rnn = nn.GRU(input_size=self.input_size * self.c_in, hidden_size=self.hidden_size, num_layers=num_rnn_layers, bias=True, batch_first=True, dropout=rnn_dropout, bidirectional=self.bidirectional)
        else:
            self.rnn = nn.LSTM(input_size=self.input_size * self.c_in, hidden_size=self.hidden_size, num_layers=num_rnn_layers, bias=True, batch_first=True, dropout=rnn_dropout, bidirectional=self.bidirectional)

        f = 2 if self.bidirectional else 1

        if mlp_final_head:
            self.linear_head = nn.Sequential(
                nn.Linear(hidden_size*f, hidden_size*f//2),
                nn.LayerNorm([hidden_size*f//2]),
                nn.ReLU(),
                nn.Dropout(linear_dropout),
                nn.Linear(hidden_size*f//2, n_classes)
            )
        else:
            self.linear_head = nn.Linear(hidden_size*f, n_classes) 
        
        if not self.predict_every_n_patches:
            self.pool_layer = nn.AdaptiveAvgPool1d(1)
        else:
            self.pool_kernel_size = int(self.predict_every_n_patches)
            self.pool_layer = nn.AvgPool1d(kernel_size=self.pool_kernel_size, stride=self.pool_kernel_size) if self.pool != 'max' else nn.MaxPool1d(kernel_size=self.pool_kernel_size, stride=self.pool_kernel_size)

        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x, return_softmax=False):
        """
        GRU input expects [bs x seq_len x H_in]
        x: new: [bs * c_in x n patch x d_model] previous: [bs x n channels x d model x n patches]
        """
        bs = x.size(0) // self.c_in
        was_nested = x.is_nested
        if was_nested:
            x = unflatten_dim_from_batch(x, self.c_in)
            lengths = [i.size(1) for i in x.unbind()]
            x = x.to_padded_tensor(padding=0)
        else:
            lengths = None
            x = x.reshape(bs, self.c_in, -1, self.input_size)
        x = self.norm_layer(x)
        x = x.permute(0,2,3,1)
        if self.affine:
             x = x * self.affine_weight
             x = x + self.affine_bias
        x = self.flatten(x)
        if was_nested:
            x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        x,hidden = self.rnn(x)

        if was_nested:

            x, _ = pad_packed_sequence(x, batch_first=True, padding_value=0, total_length=max(lengths))
        x = self.linear_head(x)
        x = x.transpose(1,2)
        x = self.pool_layer(x)
        if return_softmax:
            x = self.softmax(x)
        return x
     


In [ ]:
#| notest
m = RNNProbingHead(c_in=7, 
                    pool='average', 
                    input_size = 384, 
                    missing_channel_indices=None,#,[0,1],
                    bidirectional=True,
                    affine=True, 
                    hidden_size=1200,
                    module='GRU',
                    n_classes=5,
                    predict_every_n_patches=5,
                    rnn_dropout=0.,
                    num_rnn_layers=1,
                    mlp_final_head=True,
                    pre_norm=True)
x = torch.randn((4*7,960,384))
m(x, return_softmax=True).shape


torch.Size([4, 5, 192])

In [ ]:
#| notest
batch_size = 2
n_vars = 7
max_len = 480
d_model = 384

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length, d_model) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
x_nested = flatten_dim_to_batch(x_nested, dim=1)  # flatten n_vars into batch dimension
y = m(x_nested, return_softmax=False)
y.shape

In [ ]:
#| export
class CrossAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=12,
        qkv_bias=False,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_K = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_V = nn.Linear(dim, dim, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, q, x):
        q = self.W_Q(q).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        k = self.W_K(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        v = self.W_V(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH], set_priority=True):
            q = F.scaled_dot_product_attention(q, k, v)
        q = q.transpose(1, 2).flatten(-2)
        q = self.proj(q)
        return q


class CrossAttentionBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.,
        qkv_bias=False,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm
    ):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.xattn = CrossAttention(dim, num_heads=num_heads, qkv_bias=qkv_bias)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer)

    def forward(self, q, x):
        y = self.xattn(q, self.norm1(x))
        q = q + y
        q = q + self.mlp(self.norm2(q))
        return q

class AttentivePooler(nn.Module):
    """ Attentive Pooler """
    def __init__(
        self,
        num_queries=1,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1,
        norm_layer=nn.LayerNorm,
        init_std=0.02,
        qkv_bias=True,
        complete_block=True,
    ):
        super().__init__()
        self.query_tokens = nn.Parameter(torch.zeros(1, num_queries, embed_dim))

        self.complete_block = complete_block
        if complete_block:
            self.cross_attention_block = CrossAttentionBlock(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                norm_layer=norm_layer)
        else:
            self.cross_attention_block = CrossAttention(
                dim=embed_dim,
                num_heads=num_heads,
                qkv_bias=qkv_bias)

        self.blocks = None
        if depth > 1:
            self.blocks = nn.ModuleList([
                JEPABlock(
                    dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    qk_scale=False,
                    norm_layer=norm_layer)
                for i in range(depth-1)])

        self.init_std = init_std
        trunc_normal_(self.query_tokens, std=self.init_std)
        self.apply(self._init_weights)
        self._rescale_blocks()

    def _rescale_blocks(self):
        def rescale(param, layer_id):
            param.div_(math.sqrt(2.0 * layer_id))

        if self.complete_block:
            rescale(self.cross_attention_block.xattn.proj.weight.data, 1)
            rescale(self.cross_attention_block.mlp.fc2.weight.data, 1)
        else:
            rescale(self.cross_attention_block.proj.weight.data, 1)
        if self.blocks is not None:
            for layer_id, layer in enumerate(self.blocks, 1):
                rescale(layer.attn.proj.weight.data, layer_id + 1)
                rescale(layer.mlp.fc2.weight.data, layer_id + 1)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=self.init_std)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            trunc_normal_(m.weight, std=self.init_std)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        q = self.query_tokens.repeat(len(x), 1, 1)
        if x.is_nested:
            q = torch.nested.as_nested_tensor(q, layout=torch.jagged)
        q = self.cross_attention_block(q, x)
        if self.blocks is not None:
            for blk in self.blocks:
                q = blk(q)
        return q

class AttentiveClassifier(nn.Module):
    """ Attentive Classifier """
    def __init__(
        self,
        embed_dim=768,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1,
        norm_layer=nn.LayerNorm,
        init_std=0.02,
        qkv_bias=True,
        num_classes=1000,
        complete_block=True,
        num_queries=1,
        affine=False,
        c_in=7,
        per_channel=False
    ):
        super().__init__()
        if per_channel:
             self.poolers = nn.ModuleList([
                AttentivePooler(
                    num_queries=num_queries,
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    depth=depth,
                    norm_layer=norm_layer,
                    init_std=init_std,
                    qkv_bias=qkv_bias,
                    complete_block=complete_block,
                ) for _ in range(c_in)
            ])
        else:
            self.pooler = AttentivePooler(
                num_queries=num_queries,
                embed_dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                depth=depth,
                norm_layer=norm_layer,
                init_std=init_std,
                qkv_bias=qkv_bias,
                complete_block=complete_block,
            )
        self.c_in = c_in
        self.per_channel = per_channel

        if per_channel:
            self.linears = nn.ModuleList(nn.Linear(embed_dim, num_classes) for _ in range(c_in))
        else:
            self.linear = nn.Linear(embed_dim*c_in, num_classes, bias=True)

    def forward(self, x):
        """
        x: [bs * nvars x num_patch x d_model] 
        out: [bs x n_classes]
        """
        B_cin, n_patches, d_model = x.shape
        bs = B_cin // self.c_in
        if not self.per_channel:
            x = self.pooler(x)
            if x.is_nested:
                x_list = list(x.unbind())
                restored_list = []
                for i in range(0, len(x_list), self.c_in):
                    group = x_list[i:i+self.c_in]
                    stacked = torch.stack(group, dim=0)
                    restored_list.append(stacked.squeeze(1))
                x = torch.stack(restored_list, dim=0)
                x = x.transpose(1,2)
            else:
                x = torch.reshape(x, (-1, self.c_in, d_model))
            x = x.flatten(start_dim=-2)
            x = self.linear(x)
        else:
            if not x.is_nested:
                x = x.reshape(bs, self.c_in, n_patches, d_model)
            else:
                x = unflatten_dim_from_batch(x, self.c_in)
            x_out = []
            for i in range(self.c_in):
                for b in range(bs):
                    x_chan = x[b, i, :, :].unsqueeze(0)
                    pooled = self.poolers[i](x_chan).squeeze(1)
                    if b == 0:
                        pooled_all = pooled
                    else:
                        pooled_all = torch.cat((pooled_all, pooled), dim=0)
                pred = self.linears[i](pooled_all)
                x_out.append(pred)
            x = torch.stack(x_out, dim=1)
            x = x.mean(dim=1)

        return x
    


In [ ]:
#| notest
#  [bs x nvars x d_model x num_patch] 
x = torch.randn(4 * 7, 50, 256)
cls = AttentiveClassifier(embed_dim=256, c_in=7, num_queries=1, num_heads=2, mlp_ratio=4.0, depth=1, norm_layer=nn.LayerNorm, init_std=0.02, qkv_bias=True, num_classes=1, complete_block=False, per_channel=False)
cls(x).shape

torch.Size([28, 1, 256])


torch.Size([4, 1])

In [ ]:
#| notest
#  [bs x nvars x d_model x num_patch] 

cls_ = AttentiveClassifier(embed_dim=768, num_queries=1, num_heads=8, mlp_ratio=4.0, depth=1, norm_layer=nn.LayerNorm, init_std=0.02, qkv_bias=True, num_classes=5, complete_block=False)
sum(p.numel() for p in cls_.parameters())

2390021

In [ ]:
#| notest
#  [bs x nvars x d_model x num_patch] 
x = torch.randn(4, 7*10, 256)
cls_ = AttentiveClassifierNoMelt(embed_dim=256, num_queries=1, num_heads=2, mlp_ratio=4.0, depth=1, norm_layer=nn.LayerNorm, init_std=0.02, qkv_bias=True, num_classes=1, complete_block=True)
cls_(x).shape

torch.Size([4, 1])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()